## Setup

In [ ]:
import os
import numpy as np
import pandas as pd

In [ ]:
!cp -r /kaggle/input/datasets/theoviel/biohub-cell-tracking-code/src ./

In [ ]:
try:
    import zarr, geff, tracksdata
    
except: 
    import os
    import sys
    import subprocess
    import importlib
    import importlib.util
    from pathlib import Path
    
    os.environ.setdefault("POLARS_PREFER_PKG", "32")
    
    SUPPORT_DIR = Path(
        "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    )
    WHEELS_DIR = SUPPORT_DIR / "wheels"
    
    if not WHEELS_DIR.exists():
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find the attached offline wheels directory")
        WHEELS_DIR = candidates[0]
    
    print("Offline wheels:", WHEELS_DIR)
    
    OFFLINE_PACKAGES = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]
    
    def module_missing(module_name: str) -> bool:
        return importlib.util.find_spec(module_name) is None
    
    
    REQUIRED_IMPORTS = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }
    
    
    def purge_modules(module_roots):
        """
        Remove already-imported package modules from sys.modules.
    
        Normally this cell runs before imports, but this also protects against
        accidental imports performed by earlier Kaggle initialization code.
        """
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)
    
    
    def install_offline_packages():
        cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-index",
            "--no-deps",
            "--find-links",
            str(WHEELS_DIR),
            *OFFLINE_PACKAGES,
        ]
    
        print("Installing attached packages without modifying NumPy/SciPy/Torch...")
        result = subprocess.run(
            cmd,
            text=True,
            capture_output=True,
        )
    
        if result.returncode != 0:
            print(result.stdout[-4000:])
            print(result.stderr[-4000:])
            raise RuntimeError("Offline dependency installation failed")
    
        purge_modules(REQUIRED_IMPORTS.values())
    
    
    install_offline_packages()
    
    
    failures = {}
    
    for name, module_name in {
        **REQUIRED_IMPORTS,
        "numpy": "numpy",
        "scipy": "scipy",
        "dask": "dask.array",
        "xarray": "xarray",
    }.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    
    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )

import zarr, geff, tracksdata
print("All offline dependencies imported successfully.")

## Inference

In [ ]:
%%writefile inference.py

import sys
sys.path.append('./src/')

import warnings
warnings.filterwarnings("ignore")

import os
import json
import torch
import argparse
import numpy as np
from pathlib import Path

from tracking.main import build_model
from tracking_cellmot.io import list_datasets
from tracking.predict import PredictConfig, predict_video
from training.distributed import cleanup_distributed, init_distributed
from utils.logger import Config

DATA_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development/test/"


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--log_folder", required=True, help="run folder holding args.json and the checkpoint")
    p.add_argument("--ckpt", default="fold_fullfit/ckpt_final.pt",
                   help="checkpoint, relative to --log_folder; searched for if not found there")
    p.add_argument("--det_threshold", type=float, default=0.40,
                   help="detection threshold; pick it from the run's cv_score, not its pooled score")
    p.add_argument("--out_dir", default=None, help="default <log_folder>/preds")
    p.add_argument("--limit", type=int, default=None, help="cap the stacks (debugging)")
    p.add_argument("--data_dir", default=DATA_DIR, help="override the test split")
    return p.parse_args()


def resolve_ckpt(log_folder: Path, ckpt: str) -> Path:
    """The checkpoint, wherever the weights dataset happens to have put it.

    A run folder keeps checkpoints under fold_*/; a weights dataset is often
    flattened to the root instead. Rather than making the caller know which,
    take the path as given and fall back to a search by file name.
    """
    direct = log_folder / ckpt
    if direct.exists():
        return direct
    found = sorted(log_folder.rglob(Path(ckpt).name))
    if not found:
        available = sorted(str(f.relative_to(log_folder)) for f in log_folder.rglob("*.pt"))
        raise FileNotFoundError(
            f"{ckpt} not found under {log_folder}. Checkpoints present: {available}"
        )
    return found[0]


def main():
    args = parse_args()
    ctx = init_distributed(None)
    device = ctx.device

    paths = sorted(list_datasets(Path(args.data_dir), require_geff=False))
    if args.limit:
        paths = paths[: args.limit]

    log_folder = Path(args.log_folder)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = resolve_ckpt(log_folder, args.ckpt)

    cfg = Config(json.load(open(log_folder / "args.json")))

    # Inference settings come from the training run wherever the run fixes
    # them, so the node sets the edge head sees here are the ones it was fitted
    # against. Only the detection threshold is chosen separately -- it is swept
    # after training, and the value below is the sweep's answer.
    pred_cfg = PredictConfig
    pred_cfg.det_threshold = args.det_threshold
    pred_cfg.normalization = getattr(cfg, "normalization", None)
    pred_cfg.nms = getattr(cfg, "nms", pred_cfg.nms)
    pred_cfg.nms_radius_um = getattr(cfg, "nms_radius_um", pred_cfg.nms_radius_um)
    pred_cfg.pool_kernel_um = getattr(cfg, "pool_kernel_um", pred_cfg.pool_kernel_um)
    pred_cfg.use_offset = bool(getattr(cfg, "offset_head", False))

    window_size = getattr(cfg, "window_size", 2)
    downscale_factor = getattr(cfg, "downscale_factor", 1)

    if ctx.is_main:
        print(f"-> log_folder     : {log_folder}")
        print(f"-> checkpoint     : {ckpt_path.relative_to(log_folder)}")
        print(f"-> threshold      : {pred_cfg.det_threshold}")
        print(f"-> nms            : {pred_cfg.nms} @ {pred_cfg.nms_radius_um} um")
        print(f"-> normalization  : {pred_cfg.normalization}")
        print(f"-> window_size    : {window_size}")
        print(f"-> stacks         : {len(paths)}")
        print(f"-> writing to     : {out_dir}")
        if ctx.distributed:
            print(f"-> distributed    : {ctx.world_size} processes")
        print()

    state = torch.load(ckpt_path, map_location="cpu")
    state = state.get("model_state_dict", state)
    state = {k.replace("unet.module.", "unet.", 1): v for k, v in state.items()}

    model = build_model(cfg, None)
    model.load_state_dict(state, strict=True)
    model.eval().to(device)

    # Shard stacks across ranks; each writes its own npz files.
    mine = paths[ctx.rank :: ctx.world_size]

    for i, path in enumerate(mine):
        print(f"[rank {ctx.rank}] {i + 1}/{len(mine)}  -  {path.name}", flush=True)

        coords, edges, _ = predict_video(
            model,
            path,
            device=device,
            cfg=pred_cfg,
            window_size=window_size,
            downscale_factor=downscale_factor,
            progress=ctx.is_main,
        )

        np.savez_compressed(
            out_dir / f"{path.name}.npz",
            coords=coords,
            edges=np.asarray(edges, dtype=np.float64).reshape(-1, 4),
        )


if __name__ == "__main__":
    try:
        main()
    finally:
        cleanup_distributed()


In [ ]:
!torchrun --standalone --nproc_per_node=2 inference.py \
    --log_folder "/kaggle/input/datasets/theoviel/biohub-cell-tracking-weights/" \
    --ckpt "fold_fullfit/ckpt_final.pt" \
    --det_threshold 0.40 \
    --out_dir "/tmp/preds/"

## Post-processing

Runs `src/main_postprocess.py` directly rather than a copy kept in this
notebook. The copy had drifted: it was missing the gap-recovery stage
(worth +0.0161 on its own, +0.0063 under pp_public) and the grid-recentring
correction (+0.0065), because both were added to the repository script after
the copy was taken.

Stages: `pipeline -> recover(gap<=3) -> pp_public(tuned)`. `--no_score` because
the test split ships no ground truth.

In [ ]:
!python src/main_postprocess.py \
    --log_folder "" \
    --preds_dir "/tmp/preds/" \
    --out_dir "/tmp/preds_pp/" \
    --geff_dir "/tmp/preds_pp_geff/" \
    --no_score \
    --jobs 4

## Submission

`src/submission.py` again, rather than a copy — the two had already converged
to the same file apart from a warnings filter.

In [ ]:
!python src/submission.py /tmp/preds_pp_geff -o ./submission.csv

In [ ]:
sub = pd.read_csv('./submission.csv')
sub.sample(5)